# 📊 GMM & Mean Shift: Visualization Lab

Welcome to the GMM and Mean Shift Lab. We will visualize probability distributions, soft clustering, and density peaks.

## 1. Theory Overview

GMM models clusters as Gaussian distributions, allowing for elliptical shapes and soft assignments. Mean Shift is a density-based algorithm that climbs KDE gradients to find modes (peaks).

## 2. Visualization Playground
Let's see how GMM handles elliptical data compared to K-Means.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs
from sklearn.mixture import GaussianMixture
from matplotlib.patches import Ellipse

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Generate spherical data
X, y = make_blobs(n_samples=400, centers=4, cluster_std=0.6, random_state=42)

# Stretch it to make it elliptical
rng = np.random.RandomState(13)
X_stretched = np.dot(X, rng.randn(2, 2))

gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
labels = gmm.fit_predict(X_stretched)

def draw_ellipse(position, covariance, ax=None, **kwargs):
    ax = ax or plt.gca()
    if covariance.shape == (2, 2):
        U, s, Vt = np.linalg.svd(covariance)
        angle = np.degrees(np.arctan2(U[1, 0], U[0, 0]))
        width, height = 2 * np.sqrt(s)
    else:
        angle = 0
        width, height = 2 * np.sqrt(covariance)
    for nsig in range(1, 4):
        ax.add_patch(Ellipse(position, nsig * width, nsig * height, angle=angle, **kwargs))

plt.scatter(X_stretched[:, 0], X_stretched[:, 1], c=labels, s=20, cmap='viridis', zorder=2)
for pos, covar, w in zip(gmm.means_, gmm.covariances_, gmm.weights_):
    draw_ellipse(pos, covar, alpha=w * 0.2, color='red')

plt.title("GMM fitting Elliptical Contours")
plt.show()

## 3. From Scratch Implementation
Skipped due to complexity of matrix inversion in Python.

## 4. NumPy Implementation
N/A

## 5. Scikit-Learn Implementation (Mean Shift)
Let's visualize the KDE surface that Mean Shift climbs.

In [ ]:
from sklearn.cluster import MeanShift, estimate_bandwidth

# Using original spherical X
bandwidth = estimate_bandwidth(X, quantile=0.2, n_samples=200)
ms = MeanShift(bandwidth=bandwidth, bin_seeding=True)
ms.fit(X)
cluster_centers = ms.cluster_centers_

sns.kdeplot(x=X[:, 0], y=X[:, 1], cmap="Blues", fill=True, thresh=0.05, alpha=0.5)
plt.scatter(X[:, 0], X[:, 1], c=ms.labels_, cmap='viridis', s=10, alpha=0.5)
plt.scatter(cluster_centers[:, 0], cluster_centers[:, 1], color='red', marker='x', s=200, linewidths=3)
plt.title("Mean Shift Peaks on KDE Surface")
plt.show()

## 6. Hyperparameter Impact Study
Covariance Types in GMM (`full`, `tied`, `diag`, `spherical`).

## 7. Real Dataset Case Study
Using AIC/BIC to find the optimal number of components.

In [ ]:
n_components = np.arange(1, 10)
models = [GaussianMixture(n, covariance_type='full', random_state=42).fit(X_stretched) for n in n_components]

plt.figure(figsize=(8, 4))
plt.plot(n_components, [m.bic(X_stretched) for m in models], label='BIC', marker='o')
plt.plot(n_components, [m.aic(X_stretched) for m in models], label='AIC', marker='x')
plt.legend(loc='best')
plt.xlabel('Number of Components')
plt.ylabel('Information Criterion')
plt.title('AIC and BIC (Lower is Better)')
plt.axvline(x=4, color='r', linestyle='--')
plt.show()

## 8. Visualization Challenge
Use GMM to generate *new* synthetic data points using `.sample()`.

## 9. Exercises
1. Extract the probabilities using `predict_proba` and plot points sized proportionally to their uncertainty (points near boundaries will be larger).